<a href="https://colab.research.google.com/github/aditya01-dev/Natural-Language-Processing-Lab/blob/main/NLP_Practical_24_BERT_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Aditya kumar Singh
# ============================================================
# PRACTICAL 24: TEXT CLASSIFICATION USING PYTORCH GRU
# Course Outcome: CO3
# ============================================================

# OBJECTIVE:
# To train a recurrent neural network (Bi-GRU) for binary sentiment
# classification using PyTorch.

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import torch
import torch.nn as nn
import numpy as np
import random
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ------------------------------------------------------------
# 2. CREATE SENTIMENT DATASET
# ------------------------------------------------------------

texts = [
    "This movie was excellent and absolutely wonderful.",
    "I loved the story and the acting was fantastic.",
    "The product is amazing and works perfectly.",
    "The experience was enjoyable and satisfying.",
    "This film was brilliant and entertaining.",
    "The service was excellent and very helpful.",
    "I really enjoyed this product.",
    "The movie was beautiful and inspiring.",
    "Everything was perfect and I am very happy.",
    "The quality is outstanding and impressive.",
    "This movie was terrible and very boring.",
    "I hated the story and the acting was awful.",
    "The product is useless and completely disappointing.",
    "The experience was unpleasant and frustrating.",
    "This film was horrible and a waste of time.",
    "The service was poor and very unhelpful.",
    "I regret buying this product.",
    "The movie was dull and disappointing.",
    "Nothing worked properly and I am unhappy.",
    "The quality is bad and extremely frustrating."
]

labels = [1] * 10 + [0] * 10
# 1 = Positive, 0 = Negative

# ------------------------------------------------------------
# 3. SHUFFLE AND SPLIT DATASET
# ------------------------------------------------------------

indices = list(range(len(texts)))
random.shuffle(indices)

train_indices = indices[:16]
test_indices = indices[16:]

train_texts = [texts[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]

test_texts = [texts[i] for i in test_indices]
test_labels = [labels[i] for i in test_indices]

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

# ------------------------------------------------------------
# 4. BUILD VOCABULARY & TOKENIZER
# ------------------------------------------------------------

def tokenize(text):
    return text.lower().replace(".", "").split()

all_words = []
for text in train_texts:
    all_words.extend(tokenize(text))

vocab = ["<PAD>", "<UNK>"] + [word for word, freq in Counter(all_words).most_common()]
word2idx = {word: idx for idx, word in enumerate(vocab)}

def text_to_sequence(text, max_length=16):
    tokens = tokenize(text)
    sequence = [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]
    if len(sequence) < max_length:
        sequence += [word2idx["<PAD>"]] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]
    return sequence

# ------------------------------------------------------------
# 5. PREPARE DATALOADERS
# ------------------------------------------------------------

def prepare_data(text_list, label_list):

    sequences = [text_to_sequence(t) for t in text_list]
    sequences_tensor = torch.tensor(sequences, dtype=torch.long)
    labels_tensor = torch.tensor(label_list, dtype=torch.float32)

    return TensorDataset(sequences_tensor, labels_tensor)

train_dataset = prepare_data(train_texts, train_labels)
test_dataset = prepare_data(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# ------------------------------------------------------------
# 6. DEFINE GRU CLASSIFIER MODEL
# ------------------------------------------------------------

class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        # Concatenate forward and backward hidden states
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        logits = self.fc(hidden).squeeze(1)
        return logits

model = SentimentGRU(len(vocab), embed_dim=32, hidden_dim=16)
model.to(device)

# ------------------------------------------------------------
# 7. CONFIGURE OPTIMIZER & LOSS
# ------------------------------------------------------------

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ------------------------------------------------------------
# 8. TRAIN MODEL
# ------------------------------------------------------------

epochs = 15
print("\nTRAINING GRU MODEL")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_seqs, batch_labels in train_loader:
        batch_seqs = batch_seqs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch_seqs)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

# ------------------------------------------------------------
# 9. EVALUATE MODEL
# ------------------------------------------------------------

model.eval()
actual_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_seqs, batch_labels in test_loader:
        batch_seqs = batch_seqs.to(device)
        outputs = model(batch_seqs)
        predictions = torch.round(torch.sigmoid(outputs))

        predicted_labels.extend(predictions.cpu().tolist())
        actual_labels.extend(batch_labels.tolist())

accuracy = accuracy_score(actual_labels, predicted_labels)

print("\nMODEL EVALUATION")
print("Actual Labels:", [int(l) for l in actual_labels])
print("Predicted Labels:", [int(p) for p in predicted_labels])
print("Accuracy:", round(accuracy, 4))

# ------------------------------------------------------------
# 10. CLASSIFY NEW REVIEWS
# ------------------------------------------------------------

new_reviews = [
    "This movie was amazing and I enjoyed every moment.",
    "This product is awful and completely useless.",
    "The quality is excellent and I am very satisfied."
]

new_sequences = [text_to_sequence(r) for r in new_reviews]
new_tensor = torch.tensor(new_sequences, dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    logits = model(new_tensor)
    probabilities = torch.sigmoid(logits)
    predictions = torch.round(probabilities)

print("\nNEW REVIEW PREDICTIONS")
for review, prediction, prob in zip(new_reviews, predictions, probabilities):
    sentiment = "Positive" if prediction.item() == 1.0 else "Negative"
    confidence = prob.item() if prediction.item() == 1.0 else 1.0 - prob.item()

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print("Confidence:", round(confidence, 4))

# ------------------------------------------------------------
# 11. RESULT AND CONCLUSION
# ------------------------------------------------------------

print("\nRESULT:")
print("GRU-based text classification training completed successfully.")

print("\nCONCLUSION:")
print("An RNN classifier model was successfully built and trained using labeled text.")
print("The trained model classified reviews into positive and negative sentiment categories.")

Device: cpu
Training samples: 16
Testing samples: 4

TRAINING GRU MODEL
Epoch 1/15 | Loss: 0.6994
Epoch 5/15 | Loss: 0.5447
Epoch 10/15 | Loss: 0.0232
Epoch 15/15 | Loss: 0.0022

MODEL EVALUATION
Actual Labels: [1, 1, 1, 1]
Predicted Labels: [1, 0, 1, 0]
Accuracy: 0.5

NEW REVIEW PREDICTIONS

Review: This movie was amazing and I enjoyed every moment.
Sentiment: Positive
Confidence: 0.996

Review: This product is awful and completely useless.
Sentiment: Negative
Confidence: 0.9972

Review: The quality is excellent and I am very satisfied.
Sentiment: Positive
Confidence: 0.5024

RESULT:
GRU-based text classification training completed successfully.

CONCLUSION:
An RNN classifier model was successfully built and trained using labeled text.
The trained model classified reviews into positive and negative sentiment categories.
